# Autocorrelation of read depth along the genome

Read depth in neighbouring 500 bp windows is not independent: a high-depth window tends to sit next to other high-depth windows.
Before choosing a model for how depth changes along a chromosome, this notebook measures how far that correlation reaches.

Input: `<sample>.window_stats.tsv` in `$PF8/output/` (mean and SD of depth in each 500 bp window, from step 8 of `pf8_mosdepth.sh` or `run_samples.sh`).

Run the cells one at a time, top to bottom (`conda activate krsim`, then `jupyter lab`).

## Step 1: paths and settings

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Folder for the Pf8 data (change this on another laptop)
PF8 = "/Users/jbansal/Desktop/IMPERIAL/pf8"

WINDOW = 500      # bp per window, as made by pf8_profiles
MAX_LAG = 200     # largest lag in windows for the autocorrelation (200 windows = 100 kb)
PSEUDO = 0.05     # added before taking the log, so windows with zero depth do not give -inf

## Step 2: which samples have window stats

In [ ]:
files = sorted(glob.glob(f"{PF8}/output/*.window_stats.tsv"))
samples = [os.path.basename(f).replace(".window_stats.tsv", "") for f in files]

info = pd.read_csv(f"{PF8}/input/Pf8_samples.txt", sep="\t", index_col="Sample")
info.loc[samples, ["Country", "Year", "Population", "% callable", "Sample type"]]

## Step 3: read every sample into one table (14 nuclear chromosomes only)

In [ ]:
tables = []
for sample, f in zip(samples, files):
    df = pd.read_csv(f, sep="\t")
    df.insert(0, "sample", sample)
    tables.append(df)
depth = pd.concat(tables, ignore_index=True)

depth = depth[~depth["chrom"].str.contains("API|MIT")]   # drop the apicoplast and mitochondrion
depth

In [ ]:
# windows per chromosome (the same for every sample)
depth[depth["sample"] == samples[0]].groupby("chrom").size()

## Step 4: put the samples on the same scale

Samples were sequenced to very different depths, so each sample's window means are divided by that sample's median window mean (`rel` = 1 means a typical window).
Then `log_rel = log(rel + PSEUDO)`, because depth varies by multiples rather than by fixed amounts.

In [ ]:
median = depth.groupby("sample")["mean"].median()
depth["rel"] = depth["mean"] / depth["sample"].map(median)
depth["log_rel"] = np.log(depth["rel"] + PSEUDO)

summary = pd.DataFrame({
    "median depth": median,
    "zero windows": depth.groupby("sample")["mean"].apply(lambda m: (m == 0).sum()),
})
summary

## Step 5: look at one sample along one chromosome

In [ ]:
SAMPLE = samples[0]
CHROM = "Pf3D7_07_v3"

one = depth[(depth["sample"] == SAMPLE) & (depth["chrom"] == CHROM)]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(one["start"] / 1e3, one["rel"], lw=0.6)
axes[0].set_ylabel("depth / sample median")
axes[1].plot(one["start"] / 1e3, one["log_rel"], lw=0.6)
axes[1].set_ylabel("log(rel + PSEUDO)")
axes[1].set_xlabel(f"{CHROM} position (kb)")
axes[0].set_title(f"{SAMPLE}: mean depth in each {WINDOW} bp window")
fig.tight_layout()

## Step 6: autocorrelation function

For a series `x` along one chromosome, the autocorrelation at lag `k` is the correlation between each window and the window `k` windows further along.
It is computed separately for each chromosome, so the jump from the end of one chromosome to the start of the next is never counted.

In [ ]:
def acf(x, max_lag):
    """Autocorrelation of x at lags 0..max_lag."""
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    var = (x ** 2).sum()
    return np.array([(x[:len(x) - k] * x[k:]).sum() / var for k in range(max_lag + 1)])

## Step 7: ACF of one sample, one line per chromosome

In [ ]:
lags_kb = np.arange(MAX_LAG + 1) * WINDOW / 1e3

fig, ax = plt.subplots(figsize=(10, 5))
for chrom, g in depth[depth["sample"] == SAMPLE].groupby("chrom"):
    ax.plot(lags_kb, acf(g["log_rel"], MAX_LAG), lw=0.8, label=chrom.replace("Pf3D7_", "").replace("_v3", ""))
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("lag (kb)")
ax.set_ylabel("autocorrelation of log_rel")
ax.set_title(f"{SAMPLE}: ACF per chromosome")
ax.legend(ncol=7, fontsize=8)
fig.tight_layout()

## Step 8: ACF of every sample (averaged over the 14 chromosomes)

In [ ]:
acfs = {}
for sample in samples:
    per_chrom = [acf(g["log_rel"], MAX_LAG) for _, g in depth[depth["sample"] == sample].groupby("chrom")]
    acfs[sample] = np.mean(per_chrom, axis=0)
acfs = pd.DataFrame(acfs, index=pd.Index(lags_kb, name="lag (kb)"))

fig, ax = plt.subplots(figsize=(10, 5))
for sample in samples:
    ax.plot(lags_kb, acfs[sample], lw=1, label=f"{sample} ({info.loc[sample, 'Sample type']})")
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("lag (kb)")
ax.set_ylabel("autocorrelation of log_rel")
ax.set_title("ACF averaged over chromosomes, one line per sample")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()

In [ ]:
# the ACF at a few lags, as numbers
acfs.loc[[0.5, 1, 2, 5, 10, 25, 50, 100]].T.round(2)

## Step 9: same plot with a log y-axis

If the ACF falls as a straight line here, it decays exponentially, which is what an AR(1) process gives (ACF at lag k = φ^k).
A curve that bends upward instead means the correlation reaches further than AR(1) allows.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for sample in samples:
    ax.plot(lags_kb[1:], acfs[sample].iloc[1:].clip(lower=1e-3), lw=1, label=sample)
ax.set_yscale("log")
ax.set_xlabel("lag (kb)")
ax.set_ylabel("autocorrelation (log scale)")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()